# 03 — Training Random Forest Segmentasi Area Parkir (Google Colab)

Pipeline pelatihan **Random Forest** untuk segmentasi piksel area parkir motor (`0 = Lantai`, `1 = Motor`, `255 = Ignore`) menggunakan **111 frame bersih** dari **13 rekaman** (8 hari berbeda).

- **Anti-leakage split per hari**:
  - `train`: `2026-11-03`, `2026-11-06`, `2026-11-09`, `2026-11-17` (8 rekaman)
  - `val`: `2026-11-10`, `2026-11-18` (2 rekaman)
  - `test`: `2026-11-04`, `2026-11-12` (3 rekaman — hari uji yang sama dengan baseline repo Dinda `mIoU = 0.647`)
- **Fitur PCD**: `HSV (H_shifted, S, V_clahe)` + `CIE Lab (L, a, b)` + tekstur multi-skala (`local_std 5x5 & 11x11`, `local_mean 11x11`, `Sobel grad`, `Canny edge density 9x9`, `Laplacian energy 9x9`).
- **Evaluasi**: `IoU Floor`, `IoU Motor`, `mIoU`, `Macro-F1`, `Pixel Accuracy`, ablasi fitur, feature importance, visualisasi prediksi, dan ekspor model `.joblib`.

In [ ]:
# 1. Install / verify dependencies & extract dataset
import os, zipfile, time, joblib
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, f1_score

DATA_DIR = Path('/content/parking_data')
ZIP_CANDIDATES = [
    Path('/content/colab_dataset_111.zip'),
    Path('/content/drive/MyDrive/colab_dataset_111.zip'),
]
zip_file = next((p for p in ZIP_CANDIDATES if p.exists()), None)

if zip_file is None:
    try:
        from google.colab import files
        print('Upload colab_dataset_111.zip sekarang...')
        uploaded = files.upload()
        zip_file = Path('/content') / next(iter(uploaded.keys()))
    except ImportError:
        zip_file = Path('colab_dataset_111.zip')

DATA_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_file, 'r') as zf:
    zf.extractall(DATA_DIR)

df = pd.read_csv(DATA_DIR / 'manifest.csv')
print(f'Loaded {len(df)} frames from {zip_file}')
display(df.groupby(['split', 'date', 'weather']).size().rename('frames').reset_index())

# Verify zero date leakage
dates_by_split = {s: set(g['date']) for s, g in df.groupby('split')}
assert dates_by_split['train'].isdisjoint(dates_by_split['val']), 'Leakage train-val!'
assert dates_by_split['train'].isdisjoint(dates_by_split['test']), 'Leakage train-test!'
assert dates_by_split['val'].isdisjoint(dates_by_split['test']), 'Leakage val-test!'
print('Anti-leakage check PASSED: 0 overlapping dates across train/val/test.')

In [ ]:
# 2. PCD Feature Extraction (HSV + CLAHE + CIE Lab + Multi-scale Texture)
FEATURE_NAMES = [
    'H_shifted', 'S', 'V_clahe',
    'L_lab', 'a_lab', 'b_lab',
    'std_5', 'std_11', 'mean_11',
    'sobel_mag', 'canny_dens_9', 'lap_energy_9',
]

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def extract_pcd_features(img_bgr: np.ndarray) -> np.ndarray:
    """Return float32 feature map of shape (H, W, 12)."""
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    h_shift = ((hsv[:, :, 0].astype(np.float32) + 90.0) % 180.0) / 180.0
    s = hsv[:, :, 1].astype(np.float32) / 255.0
    v_clahe_u8 = cv2.medianBlur(clahe.apply(hsv[:, :, 2]), 5)
    v_clahe = v_clahe_u8.astype(np.float32) / 255.0

    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32) / 255.0
    L_lab, a_lab, b_lab = lab[:, :, 0], lab[:, :, 1], lab[:, :, 2]

    # Local std 5x5 and 11x11 on cleaned V
    m5 = cv2.boxFilter(v_clahe, -1, (5, 5))
    m5_sq = cv2.boxFilter(v_clahe * v_clahe, -1, (5, 5))
    std_5 = np.sqrt(np.maximum(m5_sq - m5 * m5, 0.0))

    mean_11 = cv2.boxFilter(v_clahe, -1, (11, 11))
    m11_sq = cv2.boxFilter(v_clahe * v_clahe, -1, (11, 11))
    std_11 = np.sqrt(np.maximum(m11_sq - mean_11 * mean_11, 0.0))

    # Gradient & edge texture
    gx = cv2.Sobel(v_clahe, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(v_clahe, cv2.CV_32F, 0, 1, ksize=3)
    sobel_mag = cv2.boxFilter(np.hypot(gx, gy), -1, (5, 5))

    edges = (cv2.Canny(v_clahe_u8, 60, 150) > 0).astype(np.float32)
    canny_dens_9 = cv2.boxFilter(edges, -1, (9, 9))

    lap = np.abs(cv2.Laplacian(v_clahe, cv2.CV_32F, ksize=3))
    lap_energy_9 = cv2.boxFilter(lap, -1, (9, 9))

    return np.stack([
        h_shift, s, v_clahe,
        L_lab, a_lab, b_lab,
        std_5, std_11, mean_11,
        sobel_mag, canny_dens_9, lap_energy_9,
    ], axis=-1).astype(np.float32)

print(f'Feature pipeline ready: {len(FEATURE_NAMES)} features -> {FEATURE_NAMES}')

In [ ]:
# 3. Stratified Pixel Sampling for Training Set
PIXELS_PER_FRAME = 5000  # 2500 floor + 2500 motor per train frame (~330k total)
rng = np.random.default_rng(42)

X_tr_list, y_tr_list = [], []
t0 = time.time()
train_df = df[df['split'] == 'train'].reset_index(drop=True)

for idx, row in train_df.iterrows():
    im = cv2.imread(str(DATA_DIR / row['image_path']))
    mk = cv2.imread(str(DATA_DIR / row['mask_path']), cv2.IMREAD_GRAYSCALE)
    feats = extract_pcd_features(im)
    for cls_id in (0, 1):
        ys, xs = np.where(mk == cls_id)
        if len(ys) == 0:
            continue
        n_take = min(PIXELS_PER_FRAME // 2, len(ys))
        pick = rng.choice(len(ys), size=n_take, replace=False)
        X_tr_list.append(feats[ys[pick], xs[pick]])
        y_tr_list.append(np.full(n_take, cls_id, dtype=np.uint8))

X_train = np.vstack(X_tr_list)
y_train = np.concatenate(y_tr_list)
print(f'Sampled X_train={X_train.shape} | floor={(y_train==0).sum():,} | motor={(y_train==1).sum():,} in {time.time()-t0:.1f}s')

In [ ]:
# 4. Train Random Forest Classifier
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=250,
    max_depth=22,
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight='balanced_subsample',
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)
print(f'Random Forest trained in {time.time()-t0:.1f}s')

# Feature Importance
imp = pd.Series(rf.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=True)
ax = imp.plot(kind='barh', figsize=(7, 4), color='#3b82f6', title='Random Forest Feature Importance')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# 5. Evaluate & Tune Threshold + Morphology on Validation Split
def predict_prob_map(model, img_bgr, roi_mask):
    feats = extract_pcd_features(img_bgr)
    prob = np.zeros(roi_mask.shape, dtype=np.float32)
    ys, xs = np.where(roi_mask)
    if len(ys) == 0:
        return prob
    # Predict in chunks to keep peak RAM low
    X_valid = feats[ys, xs]
    probs = np.empty(len(ys), dtype=np.float32)
    chunk = 100_000
    for s in range(0, len(ys), chunk):
        probs[s:s+chunk] = model.predict_proba(X_valid[s:s+chunk])[:, 1]
    prob[ys, xs] = probs
    return prob

def apply_morphology(pred_bin, roi_mask, ksize=7):
    if ksize <= 0:
        return pred_bin
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize, ksize))
    cleaned = cv2.morphologyEx(pred_bin.astype(np.uint8), cv2.MORPH_OPEN, k)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, k)
    return (cleaned > 0).astype(np.uint8) * roi_mask.astype(np.uint8)

def compute_metrics(y_true_list, y_pred_list):
    yt = np.concatenate(y_true_list)
    yp = np.concatenate(y_pred_list)
    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    iou_floor = tn / max(tn + fp + fn, 1)
    iou_motor = tp / max(tp + fp + fn, 1)
    miou = 0.5 * (iou_floor + iou_motor)
    acc = (tn + tp) / max(cm.sum(), 1)
    mf1 = f1_score(yt, yp, average='macro')
    return {'IoU_Floor': iou_floor, 'IoU_Motor': iou_motor, 'mIoU': miou, 'Macro_F1': mf1, 'Pixel_Acc': acc}

# Precompute probability maps on val
val_df = df[df['split'] == 'val'].reset_index(drop=True)
val_cache = []
for _, row in val_df.iterrows():
    im = cv2.imread(str(DATA_DIR / row['image_path']))
    mk = cv2.imread(str(DATA_DIR / row['mask_path']), cv2.IMREAD_GRAYSCALE)
    roi = mk != 255
    val_cache.append((mk, roi, predict_prob_map(rf, im, roi)))

best_cfg = None
rows_val = []
for thr in [0.40, 0.45, 0.50, 0.55, 0.60]:
    for ksize in [0, 7, 11]:
        yt_l, yp_l = [], []
        for mk, roi, prob in val_cache:
            pred = apply_morphology((prob >= thr).astype(np.uint8), roi, ksize=ksize)
            yt_l.append(mk[roi])
            yp_l.append(pred[roi])
        m = compute_metrics(yt_l, yp_l)
        rows_val.append({'thr': thr, 'morph_k': ksize, **m})
        if best_cfg is None or m['mIoU'] > best_cfg['mIoU']:
            best_cfg = {'thr': thr, 'morph_k': ksize, **m}

res_val = pd.DataFrame(rows_val).sort_values('mIoU', ascending=False)
display(res_val.head(8))
print('Best Val Config:', best_cfg)

In [ ]:
# 6. Final Held-Out Test Evaluation (2026-11-04 & 2026-11-12) + Visual Comparison
test_df = df[df['split'] == 'test'].reset_index(drop=True)
thr = best_cfg['thr']
ksize = best_cfg['morph_k']

yt_all, yp_all = [], []
per_rec = {}
samples_vis = []

for idx, row in test_df.iterrows():
    im = cv2.imread(str(DATA_DIR / row['image_path']))
    mk = cv2.imread(str(DATA_DIR / row['mask_path']), cv2.IMREAD_GRAYSCALE)
    roi = mk != 255
    prob = predict_prob_map(rf, im, roi)
    pred = apply_morphology((prob >= thr).astype(np.uint8), roi, ksize=ksize)

    yt_all.append(mk[roi])
    yp_all.append(pred[roi])
    rec = row['recording_id']
    per_rec.setdefault(rec, ([], []))
    per_rec[rec][0].append(mk[roi])
    per_rec[rec][1].append(pred[roi])

    if row['timestamp'] == '00-45':
        samples_vis.append((row['frame_id'], im, mk, pred))

overall = compute_metrics(yt_all, yp_all)
print('=== FINAL TEST RESULTS (Held-Out Days: 2026-11-04 & 2026-11-12) ===')
for k, v in overall.items():
    print(f'  {k:10s}: {v:.4f}')
print('  (Baseline Naive Bayes repo Dinda: mIoU=0.647, IoU_Floor=0.537, IoU_Motor=0.757, Pixel_Acc=0.811)')

print('\nPer-Video Test Breakdown:')
for rec, (yt_l, yp_l) in sorted(per_rec.items()):
    m = compute_metrics(yt_l, yp_l)
    print(f"  {rec:40s} | mIoU={m['mIoU']:.4f} | Floor={m['IoU_Floor']:.4f} | Motor={m['IoU_Motor']:.4f} | Acc={m['Pixel_Acc']:.4f}")

# Visualize Ground Truth vs RF Prediction
def overlay(im_bgr, mask_u8):
    out = im_bgr[:, :, ::-1].astype(np.float32)
    colors = {0: (60, 230, 151), 1: (255, 86, 111), 255: (112, 146, 224)}
    for v, rgb in colors.items():
        sel = mask_u8 == v
        if np.any(sel):
            out[sel] = out[sel] * 0.55 + np.array(rgb, dtype=np.float32) * 0.45
    return out.astype(np.uint8)

fig, axes = plt.subplots(len(samples_vis), 2, figsize=(14, 4.5 * len(samples_vis)))
if len(samples_vis) == 1:
    axes = [axes]
for ax_row, (fid, im, mk, pred) in zip(axes, samples_vis):
    pred_full = np.where(mk == 255, 255, pred).astype(np.uint8)
    ax_row[0].imshow(overlay(im, mk)); ax_row[0].set_title(f'GT: {fid}'); ax_row[0].axis('off')
    ax_row[1].imshow(overlay(im, pred_full)); ax_row[1].set_title(f'RF Pred (thr={thr}, k={ksize})'); ax_row[1].axis('off')
plt.tight_layout()
plt.savefig('/content/rf_test_comparison.png', dpi=150)
plt.show()

# Save trained model bundle
bundle = {
    'model': rf,
    'feature_names': FEATURE_NAMES,
    'threshold': thr,
    'morph_ksize': ksize,
    'val_metrics': best_cfg,
    'test_metrics': overall,
}
joblib.dump(bundle, '/content/rf_parking_model.joblib', compress=3)
print('Saved /content/rf_parking_model.joblib')